In [2]:
import pandas as pd
from sklearn.ensemble import IsolationForest


In [3]:
# Load the CSV file into a DataFrame
data = pd.read_csv('../data/data_uncorr.csv')

# Display the first few rows of the DataFrame
data.head()

,Unnamed: 0,name,market,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,...,round_B,round_C,round_D,round_E,round_F,round_G,international,european_or_international,time_to_first_funding,status_encoded
0,0,#waywire,News,1750000.0,acquired,USA,NY,New York City,New York,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,international,0.079452,0
1,1,'Rock' Your Paper,Publishing,40000.0,operating,EST,other,Tallinn,Tallinn,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,european,-0.213699,2
2,2,(In)Touch Network,Electronics,1500000.0,operating,GBR,other,London,London,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,european,0.000000,2
3,3,-R- Ranch and Mine,Tourism,60000.0,operating,USA,TX,Dallas,Fort Worth,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,international,0.624658,2
4,4,0-6.com,Curated Web,2000000.0,operating,other,other,other,other,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,international,1.213699,2


# One hot encoding

#### Binary Representation of Funding Rounds

In [4]:
data['had_round_A'] = [0 if x==0 else 1 for x in data['round_A']]
data['had_round_B'] = [0 if x==0 else 1 for x in data['round_B']]
data['had_round_C'] = [0 if x==0 else 1 for x in data['round_C']]
data['had_round_D'] = [0 if x==0 else 1 for x in data['round_D']]
data['had_round_E'] = [0 if x==0 else 1 for x in data['round_E']]
data['had_round_F'] = [0 if x==0 else 1 for x in data['round_F']]
data['had_round_G'] = [0 if x==0 else 1 for x in data['round_G']]
data['had_venture'] = [0 if x==0 else 1 for x in data['venture']]
data['had_seed'] = [0 if x==0 else 1 for x in data['seed']]
data['had_eq_crowdfunding'] = [0 if x==0 else 1 for x in data['equity_crowdfunding']]
data['had_pd_crowdfunding'] = [0 if x==0 else 1 for x in data['product_crowdfunding']]
data['had_angel'] = [0 if x==0 else 1 for x in data['angel']]
data['had_grant'] = [0 if x==0 else 1 for x in data['grant']]
data['had_pe'] = [0 if x==0 else 1 for x in data['private_equity']]
data['had_convert'] = [0 if x==0 else 1 for x in data['convertible_note']]

In [5]:
data.columns

Index(['Unnamed: 0', 'name', 'market', 'funding_total_usd', 'status',
       'country_code', 'state_code', 'region', 'city', 'funding_rounds',
       'founded_month', 'founded_year', 'seed', 'venture',
       'equity_crowdfunding', 'undisclosed', 'convertible_note',
       'debt_financing', 'angel', 'grant', 'private_equity',
       'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D',
       'round_E', 'round_F', 'round_G', 'international',
       'european_or_international', 'time_to_first_funding', 'status_encoded',
       'had_round_A', 'had_round_B', 'had_round_C', 'had_round_D',
       'had_round_E', 'had_round_F', 'had_round_G', 'had_venture', 'had_seed',
       'had_eq_crowdfunding', 'had_pd_crowdfunding', 'had_angel', 'had_grant',
       'had_pe', 'had_convert'],
      dtype='object')

### Market Groups

In [6]:
# Define the mapping of markets to market groups
market_groups_mapping = {
    "Software & Tecnologie Digitali": [
        "Software", "Mobile", "Enterprise Software", "SaaS", "Cloud Computing",
        "Analytics", "AI & Machine Learning", "Data Security", "IT & Cybersecurity"
    ],
    "Salute, Benessere & Biotech": [
        "Health Care", "Health and Wellness", "Medical", "Pharmaceuticals",
        "Medical Devices", "Mobile Health", "Bioinformatics", "Healthcare Services"
    ],
    "E-Commerce & Retail": [
        "E-Commerce", "Online Shopping", "Mobile Commerce", "Retail",
        "Marketplaces", "Consumer Goods", "Fashion", "Mobile Advertising"
    ],
    "Media, Comunicazione & Intrattenimento": [
        "Social Media", "Social Network Media", "Social Media Marketing",
        "Entertainment", "Music", "Video Streaming", "Social Games", "Online Video Advertising"
    ],
    "Finanza & Business": [
        "Finance", "Financial Services", "Payments", "Venture Capital",
        "Crowdfunding", "Personal Finance", "Investment Management", "Business Analytics"
    ],
    "Educazione & Formazione": [
        "Education", "K-12 Education", "Career Management", "Training",
        "Language Learning", "Tutoring"
    ],
    "Tecnologie Innovative & Energia": [
        "Clean Technology", "Renewable Energies", "Solar", "Energy Efficiency",
        "Energy IT", "Smart Grid"
    ],
    "Automotive & Trasporti": [
        "Automotive", "Transportation", "Cars", "Fleet Management", "Taxis"
    ],
    "Real Estate & Immobili": [
        "Real Estate", "Property Management", "Commercial Real Estate", "Residential Solar"
    ],
    "Diversi & Altri Settori": [
        "Startups", "Consulting", "Nonprofits", "Sports", "Events",
        "Crowdsourcing", "Art", "Charity"
    ]
}

# Create a reverse mapping for easier lookup
reverse_mapping = {market: group for group, markets in market_groups_mapping.items() for market in markets}

# Add the "market groups" column to the dataframe
data["market groups"] = data["market"].map(reverse_mapping).fillna("Unknown")

In [7]:
data["market groups"].value_counts()

market groups
Unknown                                   18810
Software & Tecnologie Digitali             6366
E-Commerce & Retail                        2056
Salute, Benessere & Biotech                1656
Media, Comunicazione & Intrattenimento     1401
Diversi & Altri Settori                     877
Finanza & Business                          844
Tecnologie Innovative & Energia             716
Educazione & Formazione                     686
Real Estate & Immobili                      340
Automotive & Trasporti                      288
Name: count, dtype: int64

In [8]:
# Perform one-hot encoding on the 'market groups' column
market_groups_encoded = pd.get_dummies(data['market groups'], prefix='market_group')

# Concatenate the encoded columns back to the original dataframe
data = pd.concat([data, market_groups_encoded], axis=1)

# Convert True/False to 0/1 for all boolean columns
market_groups_encoded = market_groups_encoded.astype(int)

# Aggiorna il dataframe originale
data.update(market_groups_encoded)

/tmp/ipykernel_146072/1548527883.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  data.update(market_groups_encoded)
/tmp/ipykernel_146072/1548527883.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  data.update(market_groups_encoded)
/tmp/ipykernel_146072/1548527883.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  data.update(market_groups_encoded)
/tmp/ipykernel_146072/1548527883.py:11: FutureWarning: Setting an item of incompatible dtype is depr

In [9]:
data.columns

Index(['Unnamed: 0', 'name', 'market', 'funding_total_usd', 'status',
       'country_code', 'state_code', 'region', 'city', 'funding_rounds',
       'founded_month', 'founded_year', 'seed', 'venture',
       'equity_crowdfunding', 'undisclosed', 'convertible_note',
       'debt_financing', 'angel', 'grant', 'private_equity',
       'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D',
       'round_E', 'round_F', 'round_G', 'international',
       'european_or_international', 'time_to_first_funding', 'status_encoded',
       'had_round_A', 'had_round_B', 'had_round_C', 'had_round_D',
       'had_round_E', 'had_round_F', 'had_round_G', 'had_venture', 'had_seed',
       'had_eq_crowdfunding', 'had_pd_crowdfunding', 'had_angel', 'had_grant',
       'had_pe', 'had_convert', 'market groups',
       'market_group_Automotive & Trasporti',
       'market_group_Diversi & Altri Settori',
       'market_group_E-Commerce & Retail',
       'market_group_Educazione & Formazione',
    

In [10]:
# Define numerical columns
numerical_columns = data.select_dtypes(include=['float64', 'int64']).columns

# Initialize the Isolation Forest model
iso_forest = IsolationForest(random_state=42, contamination=0.05)

# Fit the model and predict anomalies
isolation_forest_pred = iso_forest.fit_predict(data[numerical_columns])

# Outliers are marked as -1
outliers_isolation_forest = data[numerical_columns][isolation_forest_pred == -1]

# Calculate the percentage of outliers
outlier_percentage = (len(outliers_isolation_forest) / len(data)) * 100
print(f"Percentage of outliers: {outlier_percentage:.2f}%")

Percentage of outliers: 5.00%


In [11]:
# Filter out the outliers
data_no_outliers = data[isolation_forest_pred != -1]

# Display the shape of the new dataset
print(f"Shape of dataset after removing outliers: {data_no_outliers.shape}")

Shape of dataset after removing outliers: (32338, 60)


## Decide which scaler use

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Standardize the numerical columns
data_no_outliers[numerical_columns] = scaler.fit_transform(data_no_outliers[numerical_columns])

# Display the first few rows of the standardized data
data_no_outliers.head()

In [13]:
# Uncomment to Standardize between 0 and 1, but comment the StandardScaler cell

# from sklearn.preprocessing import MinMaxScaler

# # Initialize the MinMaxScaler
# min_max_scaler = MinMaxScaler()

# # Apply MinMaxScaler to the numerical columns
# data_no_outliers[numerical_columns] = min_max_scaler.fit_transform(data_no_outliers[numerical_columns])

# # Display the first few rows of the standardized data
# data_no_outliers.head()

## Normalize

In [14]:
from sklearn.preprocessing import normalize

# Normalize the numerical columns
data_no_outliers[numerical_columns] = normalize(data_no_outliers[numerical_columns])

# Display the first few rows of the normalized data
data_no_outliers.head()

/tmp/ipykernel_146072/2953796633.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_no_outliers[numerical_columns] = normalize(data_no_outliers[numerical_columns])


,Unnamed: 0,name,market,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,...,market_group_Diversi & Altri Settori,market_group_E-Commerce & Retail,market_group_Educazione & Formazione,market_group_Finanza & Business,"market_group_Media, Comunicazione & Intrattenimento",market_group_Real Estate & Immobili,"market_group_Salute, Benessere & Biotech",market_group_Software & Tecnologie Digitali,market_group_Tecnologie Innovative & Energia,market_group_Unknown
0,0.000000e+00,#waywire,News,0.707107,acquired,USA,NY,New York City,New York,4.040609e-07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.040609e-07
1,1.766650e-05,'Rock' Your Paper,Publishing,0.706660,operating,EST,other,Tallinn,Tallinn,1.766650e-05,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.766650e-05
2,9.428086e-07,(In)Touch Network,Electronics,0.707106,operating,GBR,other,London,London,4.714043e-07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.714043e-07
3,3.534538e-05,-R- Ranch and Mine,Tourism,0.706908,operating,USA,TX,Dallas,Fort Worth,2.356359e-05,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.178179e-05
4,1.154700e-06,0-6.com,Curated Web,0.577350,operating,other,other,other,other,2.886751e-07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.886751e-07


## Logistic Regression

In [15]:
from sklearn.model_selection import train_test_split

import statsmodels.api as sm

In [16]:
X = data_no_outliers[numerical_columns.drop('Unnamed: 0')]
y = data_no_outliers['status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [17]:
# Convert y_train to numeric values (e.g., 0 and 1)
y_train_numeric = y_train.map({'operating': 1, 'acquired': 0})

# Drop any rows with NaN values in y_train_numeric
valid_indices = y_train_numeric.dropna().index
y_train_numeric = y_train_numeric.loc[valid_indices]
X_train_valid = X_train.loc[valid_indices]

# Add a constant to the features
X_train_w_intercept = sm.add_constant(X_train_valid)

# Remove columns with constant values
X_train_w_intercept = X_train_w_intercept.loc[:, (X_train_w_intercept != X_train_w_intercept.iloc[0]).any()]

# Fit the logistic regression model
est = sm.Logit(y_train_numeric.to_numpy(), X_train_w_intercept).fit()
est.summary().tables[1]

         Current function value: 1.202884
         Iterations: 35


/home/lorenzo/projects/startup-prediction/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,coef,std err,z,P>|z|,[0.025,0.975]
funding_total_usd,3.8579,0.915,4.215,0.000,2.064,5.652
funding_rounds,1.924e+04,5166.875,3.725,0.000,9117.986,2.94e+04
founded_month,728.4232,292.364,2.491,0.013,155.400,1301.447
founded_year,586.6670,247.490,2.370,0.018,101.595,1071.739
seed,0.3432,0.921,0.373,0.710,-1.462,2.149
venture,-0.1793,0.921,-0.195,0.846,-1.984,1.626
equity_crowdfunding,3.5072,1.833,1.913,0.056,-0.086,7.100
undisclosed,0.4966,1.028,0.483,0.629,-1.518,2.511
convertible_note,1.8453,1.382,1.335,0.182,-0.863,4.554
debt_financing,-0.1316,0.947,-0.139,0.889,-1.988,1.725


## Random Forest

In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize the Random Forest classifier
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)

# Train the model
rf_model.fit(X_train_valid, y_train_numeric)

# Make predictions on the test set
y_pred = rf_model.predict(X_test)

# Map y_test to numeric values
y_test_mapped = y_test.map({'operating': 1, 'acquired': 0})

# Drop NaN values from y_test_mapped and reset its index
y_test_mapped = y_test_mapped.dropna().reset_index(drop=True)

# Align y_pred with the reset indices of y_test_mapped
y_pred_valid = y_pred[:len(y_test_mapped)]

# Evaluate the model
print("Accuracy:", accuracy_score(y_test_mapped, y_pred_valid))
print("\nClassification Report:\n", classification_report(y_test_mapped, y_pred_valid))

Accuracy: 0.8798163633382522

Classification Report:
               precision    recall  f1-score   support

         0.0       0.06      0.05      0.06       404
         1.0       0.93      0.94      0.94      5695

    accuracy                           0.88      6099
   macro avg       0.50      0.50      0.50      6099
weighted avg       0.88      0.88      0.88      6099



## Alberi decisionali

In [19]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize the Decision Tree classifier
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model
dt_model.fit(X_train_valid, y_train_numeric)

# Make predictions on the test set
y_pred_dt = dt_model.predict(X_test)

# Align y_pred_dt with the reset indices of y_test_mapped
y_pred_dt_valid = y_pred_dt[:len(y_test_mapped)]

# Evaluate the model
print("Accuracy:", accuracy_score(y_test_mapped, y_pred_dt_valid))
print("\nClassification Report:\n", classification_report(y_test_mapped, y_pred_dt_valid))

Accuracy: 0.8798163633382522

Classification Report:
               precision    recall  f1-score   support

         0.0       0.06      0.05      0.06       404
         1.0       0.93      0.94      0.94      5695

    accuracy                           0.88      6099
   macro avg       0.50      0.50      0.50      6099
weighted avg       0.88      0.88      0.88      6099



## Modello nearest neighbors

In [20]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize the K-Nearest Neighbors classifier
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train the model
knn_model.fit(X_train_valid, y_train_numeric)

# Make predictions on the test set
y_pred_knn = knn_model.predict(X_test)

# Align y_pred_knn with the reset indices of y_test_mapped
y_pred_knn_valid = y_pred_knn[:len(y_test_mapped)]

# Evaluate the model
print("Accuracy:", accuracy_score(y_test_mapped, y_pred_knn_valid))
print("\nClassification Report:\n", classification_report(y_test_mapped, y_pred_knn_valid))

Accuracy: 0.9263813739957371

Classification Report:
               precision    recall  f1-score   support

         0.0       0.04      0.00      0.01       404
         1.0       0.93      0.99      0.96      5695

    accuracy                           0.93      6099
   macro avg       0.49      0.50      0.49      6099
weighted avg       0.87      0.93      0.90      6099

